# Text Preprocessing Pipeline

Load the raw ABSA dataset and inspect its structure.

In [1]:
import pandas as pd

absa_df = pd.read_csv("../data/raw/absa_dataset.csv")

print("Shape:", absa_df.shape)
absa_df.head()

Shape: (13100, 4)


,feedback_id,feedback_text,aspect,sentiment
0,fb_00001,Network coverage is absolutely pathetic in my ...,Network Coverage,Negative
1,fb_00001,Network coverage is absolutely pathetic in my ...,Call Quality,Negative
2,fb_00002,fr superb internet speed after the recent upgr...,Internet Speed,Positive
3,fb_00003,The network outage last week was terrible. No ...,Network Outage,Negative
4,fb_00003,The network outage last week was terrible. No ...,Call Quality,Positive


Check for missing values.

In [2]:
absa_df.isnull().sum()

feedback_id      0
feedback_text    0
aspect           0
sentiment        0
dtype: int64

Define the text cleaning function (lowercase, remove URLs/emails/special chars, collapse whitespace).

In [3]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

Apply text cleaning and verify results.

In [4]:
absa_df["clean_text"] = absa_df["feedback_text"].apply(clean_text)
absa_df[["feedback_text", "clean_text"]].head()

,feedback_text,clean_text
0,Network coverage is absolutely pathetic in my ...,network coverage is absolutely pathetic in my ...
1,Network coverage is absolutely pathetic in my ...,network coverage is absolutely pathetic in my ...
2,fr superb internet speed after the recent upgr...,fr superb internet speed after the recent upgr...
3,The network outage last week was terrible. No ...,the network outage last week was terrible no s...
4,The network outage last week was terrible. No ...,the network outage last week was terrible no s...


Tokenize the cleaned text using NLTK word_tokenize.

In [5]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize

absa_df["tokens"] = absa_df["clean_text"].apply(word_tokenize)

absa_df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,network coverage is absolutely pathetic in my ...,"[network, coverage, is, absolutely, pathetic, ..."
1,network coverage is absolutely pathetic in my ...,"[network, coverage, is, absolutely, pathetic, ..."
2,fr superb internet speed after the recent upgr...,"[fr, superb, internet, speed, after, the, rece..."
3,the network outage last week was terrible no s...,"[the, network, outage, last, week, was, terrib..."
4,the network outage last week was terrible no s...,"[the, network, outage, last, week, was, terrib..."


Remove stopwords, retaining negation words (not, no, nor, never) for sentiment analysis.

In [6]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
keep_words = {"not", "no", "nor", "never"}
stop_words = stop_words - keep_words

absa_df["tokens"] = absa_df["tokens"].apply(
    lambda words: [
        word for word in words
        if word not in stop_words
    ]
)

absa_df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,network coverage is absolutely pathetic in my ...,"[network, coverage, absolutely, pathetic, area..."
1,network coverage is absolutely pathetic in my ...,"[network, coverage, absolutely, pathetic, area..."
2,fr superb internet speed after the recent upgr...,"[fr, superb, internet, speed, recent, upgrade,..."
3,the network outage last week was terrible no s...,"[network, outage, last, week, terrible, no, si..."
4,the network outage last week was terrible no s...,"[network, outage, last, week, terrible, no, si..."


Apply lemmatization to reduce words to their base forms.

In [7]:
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

absa_df["tokens"] = absa_df["tokens"].apply(
    lambda words: [
        lemmatizer.lemmatize(word)
        for word in words
    ]
)

absa_df["tokens"].head()

0    [network, coverage, absolutely, pathetic, area...
1    [network, coverage, absolutely, pathetic, area...
2    [fr, superb, internet, speed, recent, upgrade,...
3    [network, outage, last, week, terrible, no, si...
4    [network, outage, last, week, terrible, no, si...
Name: tokens, dtype: object

Join tokens into processed_text and sample outputs for verification.

In [8]:
absa_df["processed_text"] = absa_df["tokens"].apply(lambda x: " ".join(x))

absa_df[["feedback_text", "processed_text"]].sample(10, random_state=42)

,feedback_text,processed_text
4347,Support on WhatsApp is responsive and efficien...,support whatsapp responsive efficient network ...
7732,Postpaid plan with family sharing is well desi...,postpaid plan family sharing well designed por...
3545,SMS services are basic but functional. Network...,sm service basic functional network congestion...
1237,Number portability process was painful. SMS co...,number portability process painful sm code nev...
720,SIM activation failed online twice. Had to vis...,sim activation failed online twice visit store...
8522,Value for money is excellent. Best in the mark...,value money excellent best market sim activati...
9313,Billing is okay. Clear enough to understand.,billing okay clear enough understand
9909,SERIOUSLY Billing dizpute took weeks to resolv...,seriously billing dizpute took week resolve cu...
1739,Device compatibility with my phone is perfect....,device compatibility phone perfect ott bundle ...
2452,Data balance is generous and internet speed is...,data balance generous internet speed fast grea...


Remove short tokens (≤2 chars) that are typically slang filler, then drop any empty rows.

In [9]:
# Remove tokens <= 2 characters. After stopword removal, remaining
# short tokens (e.g. "fr", "u", "ur") are almost always slang filler.
# This is maintainable and catches current + future short slang.
IMPORTANT_SHORT_WORDS = {
    "3g",
    "4g",
    "5g"
}

MIN_TOKEN_LENGTH = 3

absa_df["tokens"] = absa_df["tokens"].apply(
    lambda words: [
        word for word in words
        if len(word) >= MIN_TOKEN_LENGTH
        or word in IMPORTANT_SHORT_WORDS
    ]
)

absa_df["processed_text"] = absa_df["tokens"].apply(
    lambda words: " ".join(words)
)

absa_df = absa_df[
    absa_df["processed_text"].str.strip() != ""
]

Save the processed dataset to CSV.

In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)

absa_df.to_csv(
    "../data/processed/absa_processed.csv",
    index=False
)

print(f"Saved processed dataset: {absa_df.shape}")

Saved processed dataset: (13100, 7)
